In [11]:
import json
import asyncio
import sys
import sqlite3

from pathlib import Path

# Walk up from CWD to find the project root (identified by pyproject.toml),
# so imports work regardless of where Jupyter is launched from.
def find_project_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise RuntimeError(f"Could not find project root from {start}")

project_root = find_project_root(Path().resolve())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from movie_ingestion.metadata_generation.inputs import MovieInputData
from movie_ingestion.metadata_generation.generators.plot_events import generate_plot_events
from movie_ingestion.metadata_generation.generators.reception import generate_reception
from movie_ingestion.metadata_generation.generators.plot_analysis import generate_plot_analysis
from movie_ingestion.metadata_generation.generators.viewer_experience import generate_viewer_experience
from movie_ingestion.metadata_generation.generators.watch_context import generate_watch_context
from movie_ingestion.metadata_generation.generators.narrative_techniques import generate_narrative_techniques
from movie_ingestion.metadata_generation.generators.production_keywords import generate_production_keywords
from movie_ingestion.metadata_generation.generators.source_of_inspiration import generate_source_of_inspiration
from implementation.llms.generic_methods import LLMProvider
from movie_ingestion.tracker import deserialize_imdb_row

In [12]:
ORIGINAL_SET_TMDB_IDS = [9377, 269149, 1584, 109445, 2493, 354912, 508965, 14160, 10674, 808, 13397, 76341, 85, 155, 245891, 1771, 569094, 299534, 11, 671, 120, 98, 27205, 603, 157336, 335984, 329, 329865, 493922, 694, 49018, 1034541, 176, 807, 496243, 419430, 1359, 550, 597, 13, 666277, 423, 11036, 1824, 25195, 216015, 392044, 545611, 22538, 37136]
MEDIUM_SPARSITY_TMDB_IDS = [329974, 1498832, 821937, 92, 160, 45739, 576560, 1383243, 1642210, 1639488]
HIGH_SPARSITY_TMDB_IDS = [270909, 493103, 64262, 1611977, 706910, 1297426, 35952, 158227, 215782, 1642486]

ALL_TMDB_IDS = ORIGINAL_SET_TMDB_IDS + MEDIUM_SPARSITY_TMDB_IDS + HIGH_SPARSITY_TMDB_IDS

# Open the tracker DB directly using the project root from the setup cell,
# avoiding reliance on CWD (init_db() uses a relative path).
db = sqlite3.connect(str(project_root / "ingestion_data" / "tracker.db"))
db.row_factory = sqlite3.Row

placeholders = ",".join("?" * len(ALL_TMDB_IDS))

# Fetch title + release_date from tmdb_data (the only fields tmdb_data has
# that we need; overview text lives in imdb_data.overview, not tmdb_data).
tmdb_rows: dict[int, sqlite3.Row] = {
    row["tmdb_id"]: row
    for row in db.execute(
        f"SELECT tmdb_id, title, release_date, maturity_rating "
        f"FROM tmdb_data WHERE tmdb_id IN ({placeholders})",
        ALL_TMDB_IDS,
    ).fetchall()
}

# Fetch all imdb_data columns; deserialize_imdb_row parses JSON array columns.
imdb_rows: dict[int, dict] = {
    row["tmdb_id"]: deserialize_imdb_row(row)
    for row in db.execute(
        f"SELECT * FROM imdb_data WHERE tmdb_id IN ({placeholders})",
        ALL_TMDB_IDS,
    ).fetchall()
}

db.close()


def build_movie_input(tmdb_id: int) -> MovieInputData | None:
    tmdb = tmdb_rows.get(tmdb_id)
    if tmdb is None:
        print(f"WARNING: tmdb_id {tmdb_id} not found in tmdb_data, skipping")
        return None
    imdb = imdb_rows.get(tmdb_id, {})

    release_date = tmdb["release_date"] or ""
    release_year = int(release_date[:4]) if release_date else None

    # Prefer IMDB maturity_rating; fall back to TMDB where IMDB has none.
    maturity_rating = imdb.get("maturity_rating") or tmdb["maturity_rating"] or ""

    return MovieInputData(
        tmdb_id=tmdb_id,
        title=tmdb["title"],
        release_year=release_year,
        overview=imdb.get("overview") or "",
        genres=imdb.get("genres") or [],
        plot_synopses=imdb.get("synopses") or [],      # imdb_data column is "synopses"
        plot_summaries=imdb.get("plot_summaries") or [],
        plot_keywords=imdb.get("plot_keywords") or [],
        overall_keywords=imdb.get("overall_keywords") or [],
        featured_reviews=imdb.get("featured_reviews") or [],
        reception_summary=imdb.get("reception_summary"),
        audience_reception_attributes=imdb.get("review_themes") or [],              # not stored in the DB
        maturity_rating=maturity_rating,
        maturity_reasoning=imdb.get("maturity_reasoning") or [],
        parental_guide_items=imdb.get("parental_guide_items") or [],
    )


movies = [m for tmdb_id in ALL_TMDB_IDS if (m := build_movie_input(tmdb_id)) is not None]
print(f"Built {len(movies)} MovieInputData objects from {len(ALL_TMDB_IDS)} requested IDs")

Built 70 MovieInputData objects from 70 requested IDs


In [13]:
from dataclasses import dataclass, field as dc_field


@dataclass(frozen=True)
class PlaygroundCandidate:
    """A model configuration to test in the playground.

    Supports optional system_prompt and response_format overrides for
    experiments that need non-default prompts or schemas (e.g. the
    with-justifications variant). When not set, generator functions
    use their own internal defaults.
    """
    label: str
    provider: LLMProvider
    model: str
    kwargs: dict = dc_field(default_factory=dict)
    system_prompt: str | None = None
    response_format: type | None = None


from movie_ingestion.metadata_generation.prompts.watch_context import (
    SYSTEM_PROMPT_WITH_JUSTIFICATIONS as WC_SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
)
from movie_ingestion.metadata_generation.schemas import (
    WatchContextWithJustificationsOutput,
)

CANDIDATES = [
    PlaygroundCandidate(
        "r3-gpt5mini-low-just",
        LLMProvider.OPENAI,
        "gpt-5-mini",
        {"reasoning_effort": "low", "verbosity": "low"},
        WC_SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
        WatchContextWithJustificationsOutput,
    ),
    PlaygroundCandidate(
        "r3-gpt54nano-high-just",
        LLMProvider.OPENAI,
        "gpt-5.4-nano",
        {"reasoning_effort": "high", "verbosity": "low"},
        WC_SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
        WatchContextWithJustificationsOutput,
    ),
]

print(f"Defined {len(CANDIDATES)} candidates")

Defined 2 candidates


In [14]:
from pydantic import BaseModel


def print_all_fields(result: BaseModel) -> None:
    """Print every field of a Pydantic result on its own line.

    Solves the problem where __str__() omits certain fields
    (e.g., ReceptionOutput omits review_insights_brief).
    """
    for field_name in type(result).model_fields:
        value = getattr(result, field_name)
        if isinstance(value, list):
            print(f"  {field_name}:")
            for item in value:
                print(f"    - {item}")
        elif isinstance(value, BaseModel):
            print(f"  {field_name}:")
            for sub_field in type(value).model_fields:
                sub_value = getattr(value, sub_field)
                print(f"    {sub_field}: {sub_value}")
        else:
            print(f"  {field_name}: {value}")


async def run_candidates(movie: MovieInputData, generator_fn, extra_kwargs: dict | None = None) -> None:
    """Run all candidates through a generator function, printing results and token usage.

    Args:
        movie: The movie input data to test with.
        generator_fn: An async generator function (e.g., generate_reception).
        extra_kwargs: Additional keyword args passed to the generator before
            candidate kwargs (e.g., plot_summary, review_insights_brief).
    """
    if extra_kwargs is None:
        extra_kwargs = {}

    for candidate in CANDIDATES:
        print(f"\n{'─' * 60}")
        print(f"Model: {candidate.label}")
        try:
            result, token_usage = await generator_fn(
                movie,
                **extra_kwargs,
                provider=candidate.provider,
                model=candidate.model,
                **candidate.kwargs,
            )
            print_all_fields(result)
            print(f"  token_usage: {token_usage}")
        except Exception as e:
            print(f"  ERROR: {e}")

In [15]:
MODEL_PRICING: dict[str, tuple[float, float]] = {
    # (input_price_per_M, output_price_per_M)
    "qwen3.5-flash":                                   (0.10, 0.40),
    "gemini-2.5-flash":                                (0.30, 2.50),
    "gemini-2.5-flash-lite":                           (0.10, 0.40),
    "gpt-5-mini":                                      (0.25, 2.00),
    "gpt-5-nano":                                      (0.05, 0.40),
    "gpt-5.4-nano":                                    (0.20, 1.25),
    "openai/gpt-oss-120b":                             (0.15, 0.60),
    "meta-llama/llama-4-scout-17b-16e-instruct":       (0.11, 0.34),
    "kimi-k2.5":                                       (0.60, 3.00),
    "gemini-3.1-flash-lite-preview":                   (0.25, 1.50),
}

In [16]:
# ── Reception (Wave 1) ───────────────────────────────────────────────────────
# Inputs: reception_summary, audience_reception_attributes, featured_reviews
# Buckets: 6 groups by combined_top5_chars (review richness)

import json as _json
from movie_ingestion.metadata_generation.generators.reception import (
    build_reception_user_prompt,
    generate_reception,
)

# ── Load bucket IDs from reception_eval_buckets.json ─────────────────────────
_buckets_path = project_root / "ingestion_data" / "reception_eval_buckets.json"
with open(_buckets_path) as _f:
    _buckets = _json.load(_f)

ULTRA_THIN_0_1K = [s["tmdb_id"] for s in _buckets["ultra_thin_0_1000"]["samples"]]
VERY_THIN_1K_2500 = [s["tmdb_id"] for s in _buckets["very_thin_1000_2500"]["samples"]]
THIN_2500_5K = [s["tmdb_id"] for s in _buckets["thin_2500_5000"]["samples"]]
MODERATE_5K_7500 = [s["tmdb_id"] for s in _buckets["moderate_5000_7500"]["samples"]]
RICH_7500_10500 = [s["tmdb_id"] for s in _buckets["rich_7500_10500"]["samples"]]
VERY_RICH_10500_PLUS = [s["tmdb_id"] for s in _buckets["very_rich_10500_plus"]["samples"]]

EVAL_GROUPS = {
    "ultra_thin_0_1k": ULTRA_THIN_0_1K,
    "very_thin_1k_2500": VERY_THIN_1K_2500,
    "thin_2500_5k": THIN_2500_5K,
    "moderate_5k_7500": MODERATE_5K_7500,
    "rich_7500_10500": RICH_7500_10500,
    "very_rich_10500_plus": VERY_RICH_10500_PLUS,
}

# Load MovieInputData for all evaluation IDs
ALL_EVAL_IDS = [tid for group in EVAL_GROUPS.values() for tid in group]

db = sqlite3.connect(str(project_root / "ingestion_data" / "tracker.db"))
db.row_factory = sqlite3.Row

placeholders = ",".join("?" * len(ALL_EVAL_IDS))
eval_tmdb_rows = {
    row["tmdb_id"]: row
    for row in db.execute(
        f"SELECT tmdb_id, title, release_date, maturity_rating "
        f"FROM tmdb_data WHERE tmdb_id IN ({placeholders})",
        ALL_EVAL_IDS,
    ).fetchall()
}
eval_imdb_rows = {
    row["tmdb_id"]: deserialize_imdb_row(row)
    for row in db.execute(
        f"SELECT * FROM imdb_data WHERE tmdb_id IN ({placeholders})",
        ALL_EVAL_IDS,
    ).fetchall()
}
db.close()

def _build_eval_movie(tmdb_id: int) -> MovieInputData | None:
    tmdb = eval_tmdb_rows.get(tmdb_id)
    if tmdb is None:
        print(f"WARNING: tmdb_id {tmdb_id} not found in tmdb_data, skipping")
        return None
    imdb = eval_imdb_rows.get(tmdb_id, {})
    release_date = tmdb["release_date"] or ""
    release_year = int(release_date[:4]) if release_date else None
    maturity_rating = imdb.get("maturity_rating") or tmdb["maturity_rating"] or ""
    return MovieInputData(
        tmdb_id=tmdb_id,
        title=tmdb["title"],
        release_year=release_year,
        overview=imdb.get("overview") or "",
        genres=imdb.get("genres") or [],
        plot_synopses=imdb.get("synopses") or [],
        plot_summaries=imdb.get("plot_summaries") or [],
        plot_keywords=imdb.get("plot_keywords") or [],
        overall_keywords=imdb.get("overall_keywords") or [],
        featured_reviews=imdb.get("featured_reviews") or [],
        reception_summary=imdb.get("reception_summary"),
        audience_reception_attributes=imdb.get("review_themes") or [],
        maturity_rating=maturity_rating,
        maturity_reasoning=imdb.get("maturity_reasoning") or [],
        parental_guide_items=imdb.get("parental_guide_items") or [],
    )

eval_movies: dict[int, MovieInputData] = {}
for tid in ALL_EVAL_IDS:
    m = _build_eval_movie(tid)
    if m is not None:
        eval_movies[tid] = m
print(f"Built {len(eval_movies)} MovieInputData objects for evaluation")

# ── Use first 4 candidates ──────────────────────────────────────────────────
reception_candidates = CANDIDATES
print(f"Using {len(reception_candidates)} candidates:")
for c in reception_candidates:
    print(f"  - {c.label} ({c.model})")


def _compute_cost(model: str, input_tokens: int, output_tokens: int) -> float | None:
    """Compute generation cost in USD from MODEL_PRICING."""
    pricing = MODEL_PRICING.get(model)
    if pricing is None:
        return None
    input_price, output_price = pricing
    return (input_tokens * input_price + output_tokens * output_price) / 1_000_000


async def _generate_for_candidate(
    movie: MovieInputData,
    candidate: "PlaygroundCandidate",
) -> dict:
    """Generate reception for one (movie, candidate) pair. Returns a result dict."""
    try:
        result, token_usage = await generate_reception(
            movie,
            provider=candidate.provider,
            model=candidate.model,
            **candidate.kwargs,
        )
        cost = _compute_cost(
            token_usage.model, token_usage.input_tokens, token_usage.output_tokens,
        )
        cost_str = f"${cost:.6f}" if cost is not None else "n/a"
        print(
            f"  ✓ {candidate.label}: "
            f"{token_usage.input_tokens} in / {token_usage.output_tokens} out, "
            f"cost={cost_str}"
        )
        return {
            "candidate": candidate.label,
            "model": candidate.model,
            "result": result.model_dump(),
            "input_tokens": token_usage.input_tokens,
            "output_tokens": token_usage.output_tokens,
            "cost_usd": cost,
        }
    except Exception as e:
        print(f"  ✗ {candidate.label}: {type(e).__name__}: {e}")
        return {
            "candidate": candidate.label,
            "model": candidate.model,
            "result": None,
            "error": f"{type(e).__name__}: {e}",
            "input_tokens": None,
            "output_tokens": None,
            "cost_usd": None,
        }


def _merge_candidate_results(
    existing_results: list[dict],
    new_results: list[dict],
) -> list[dict]:
    """Merge new candidate results into existing ones.

    If a candidate with the same label already exists, replace that entry.
    Otherwise append the new entry. Preserves ordering of existing entries.
    """
    # Index existing results by label for O(1) lookup
    merged = {r["candidate"]: r for r in existing_results}
    # Overwrite or insert new results
    for r in new_results:
        merged[r["candidate"]] = r
    # Preserve original order for existing entries, append new ones at end
    seen = set()
    ordered = []
    for r in existing_results:
        label = r["candidate"]
        ordered.append(merged[label])
        seen.add(label)
    for r in new_results:
        if r["candidate"] not in seen:
            ordered.append(merged[r["candidate"]])
            seen.add(r["candidate"])
    return ordered

Built 36 MovieInputData objects for evaluation
Using 2 candidates:
  - r3-gpt5mini-low-just (gpt-5-mini)
  - r3-gpt54nano-high-just (gpt-5.4-nano)


In [17]:
# # ── Generate per-movie, all candidates concurrently ─────────────────────────

# eval_dir = project_root / "movie_ingestion" / "metadata_generation" / "evaluation_data"
# eval_dir.mkdir(parents=True, exist_ok=True)

# total_movies = len(eval_movies)
# for movie_idx, (tmdb_id, movie) in enumerate(eval_movies.items(), 1):
#     title = movie.title_with_year()
#     user_prompt = build_reception_user_prompt(movie)

#     print(f"\n[{movie_idx}/{total_movies}] {title} (tmdb_id={tmdb_id})")

#     # Fire all candidates concurrently — each hits a different provider,
#     # so concurrent calls spread rate-limit pressure across providers.
#     candidate_results = await asyncio.gather(*[
#         _generate_for_candidate(movie, c) for c in reception_candidates
#     ])

#     # Load existing file (if any) and merge new results into it
#     out_path = eval_dir / f"reception_{tmdb_id}.json"
#     if out_path.exists():
#         existing_output = _json.loads(out_path.read_text())
#         existing_candidates = existing_output.get("candidate_results", [])
#         merged = _merge_candidate_results(existing_candidates, list(candidate_results))
#     else:
#         merged = list(candidate_results)

#     movie_output = {
#         "tmdb_id": tmdb_id,
#         "title": title,
#         "user_prompt": user_prompt,
#         "candidate_results": merged,
#     }
#     out_path.write_text(_json.dumps(movie_output, indent=2, ensure_ascii=False))
#     print(f"  Saved → {out_path.name} ({len(merged)} candidates)")

# print(f"\nDone: {total_movies} movies × {len(reception_candidates)} candidates")

In [18]:
# # ── Plot Analysis Experiment 2: justification schema impact ──────────────────
# # Tests whether the with-justifications schema/prompt changes plot_analysis
# # quality compared to the standard schema, across 8 buckets.
# #
# # For each movie: 3 candidates = 3 runs.
# # All candidates omit emotional_observations and use the justification schema.
# # Results saved per-movie to evaluation_data/plot_analysis_{tmdb_id}.json.

# import json as _json
# import asyncio
# from movie_ingestion.metadata_generation.generators.plot_analysis import (
#     build_plot_analysis_user_prompt,
#     generate_plot_analysis,
# )
# from movie_ingestion.metadata_generation.schemas import (
#     PlotAnalysisOutput,
#     PlotAnalysisWithJustificationsOutput,
# )
# from movie_ingestion.metadata_generation.prompts.plot_analysis import (
#     SYSTEM_PROMPT as PA_SYSTEM_PROMPT,
#     SYSTEM_PROMPT_WITH_JUSTIFICATIONS as PA_SYSTEM_PROMPT_JUSTIFICATIONS,
# )

# # ── Candidates: 3 models, all justification schema ──────────────────────────

# @dataclass(frozen=True)
# class _PACandidate:
#     """Extends PlaygroundCandidate with schema/prompt pairing."""
#     label: str
#     provider: LLMProvider
#     model: str
#     system_prompt: str
#     response_format: type
#     kwargs: dict = dc_field(default_factory=dict)

# PLOT_ANALYSIS_CANDIDATES = [
#     _PACandidate(
#         "kimi-k2.5-no-thinking", LLMProvider.KIMI, "kimi-k2.5",
#         PA_SYSTEM_PROMPT_JUSTIFICATIONS, PlotAnalysisWithJustificationsOutput,
#         {"enable_thinking": False},
#     ),
#     _PACandidate(
#         "qwen3.5-flash-no-thinking", LLMProvider.ALIBABA, "qwen3.5-flash",
#         PA_SYSTEM_PROMPT_JUSTIFICATIONS, PlotAnalysisWithJustificationsOutput,
#         {"temperature": 0.1, "extra_body": {"enable_thinking": False}},
#     ),
#     _PACandidate(
#         "gpt-5.4-nano-medium", LLMProvider.OPENAI, "gpt-5.4-nano",
#         PA_SYSTEM_PROMPT_JUSTIFICATIONS, PlotAnalysisWithJustificationsOutput,
#         {"reasoning_effort": "medium", "verbosity": "low"},
#     ),
# ]

# # ── Load buckets and build MovieInputData ────────────────────────────────────

# _pa_buckets_path = project_root / "ingestion_data" / "plot_analysis_eval_buckets.json"
# with open(_pa_buckets_path) as _f:
#     _pa_config = _json.load(_f)

# # Collect all unique tmdb_ids across all buckets
# _pa_all_ids = list({
#     tid
#     for bucket in _pa_config["buckets"].values()
#     for tid in bucket["tmdb_ids"]
# })

# _pa_db = sqlite3.connect(str(project_root / "ingestion_data" / "tracker.db"))
# _pa_db.row_factory = sqlite3.Row

# _pa_placeholders = ",".join("?" * len(_pa_all_ids))

# # Load tmdb + imdb rows
# _pa_tmdb_rows = {
#     row["tmdb_id"]: row
#     for row in _pa_db.execute(
#         f"SELECT tmdb_id, title, release_date, maturity_rating "
#         f"FROM tmdb_data WHERE tmdb_id IN ({_pa_placeholders})",
#         _pa_all_ids,
#     ).fetchall()
# }
# _pa_imdb_rows = {
#     row["tmdb_id"]: deserialize_imdb_row(row)
#     for row in _pa_db.execute(
#         f"SELECT * FROM imdb_data WHERE tmdb_id IN ({_pa_placeholders})",
#         _pa_all_ids,
#     ).fetchall()
# }

# # Load Wave 1 outputs (plot_events + reception) from generated_metadata
# _pa_wave1 = {}
# for row in _pa_db.execute(
#     f"SELECT tmdb_id, plot_events, reception "
#     f"FROM generated_metadata WHERE tmdb_id IN ({_pa_placeholders})",
#     _pa_all_ids,
# ).fetchall():
#     _pa_wave1[row["tmdb_id"]] = {
#         "plot_events": _json.loads(row["plot_events"]) if row["plot_events"] else None,
#         "reception": _json.loads(row["reception"]) if row["reception"] else None,
#     }

# _pa_db.close()


# def _build_pa_movie(tmdb_id: int) -> MovieInputData | None:
#     tmdb = _pa_tmdb_rows.get(tmdb_id)
#     if tmdb is None:
#         print(f"WARNING: tmdb_id {tmdb_id} not found in tmdb_data, skipping")
#         return None
#     imdb = _pa_imdb_rows.get(tmdb_id, {})
#     release_date = tmdb["release_date"] or ""
#     release_year = int(release_date[:4]) if release_date else None
#     maturity_rating = imdb.get("maturity_rating") or tmdb["maturity_rating"] or ""
#     return MovieInputData(
#         tmdb_id=tmdb_id,
#         title=tmdb["title"],
#         release_year=release_year,
#         overview=imdb.get("overview") or "",
#         genres=imdb.get("genres") or [],
#         plot_synopses=imdb.get("synopses") or [],
#         plot_summaries=imdb.get("plot_summaries") or [],
#         plot_keywords=imdb.get("plot_keywords") or [],
#         overall_keywords=imdb.get("overall_keywords") or [],
#         featured_reviews=imdb.get("featured_reviews") or [],
#         reception_summary=imdb.get("reception_summary"),
#         audience_reception_attributes=imdb.get("review_themes") or [],
#         maturity_rating=maturity_rating,
#         maturity_reasoning=imdb.get("maturity_reasoning") or [],
#         parental_guide_items=imdb.get("parental_guide_items") or [],
#     )


# # Build MovieInputData + extract Wave 1 intermediates for each movie
# _pa_movies: dict[int, dict] = {}
# for tid in _pa_all_ids:
#     movie = _build_pa_movie(tid)
#     if movie is None:
#         continue
#     w1 = _pa_wave1.get(tid, {})
#     pe = w1.get("plot_events")
#     rec = w1.get("reception")
#     _pa_movies[tid] = {
#         "movie": movie,
#         "plot_synopsis": pe["plot_summary"] if pe else None,
#         "thematic_observations": rec.get("thematic_observations") if rec else None,
#     }

# print(f"Built {len(_pa_movies)} movies for plot_analysis evaluation")
# print(f"Buckets: {list(_pa_config['buckets'].keys())}")
# print(f"Candidates: {[c.label for c in PLOT_ANALYSIS_CANDIDATES]}")


# # ── Generation helpers ───────────────────────────────────────────────────────

# _PA_CONCURRENCY = asyncio.Semaphore(6)


# async def _pa_generate_one(
#     movie: MovieInputData,
#     plot_synopsis: str | None,
#     thematic_observations: str | None,
#     candidate: _PACandidate,
# ) -> dict:
#     """Generate plot_analysis for one (movie, candidate) combo."""
#     async with _PA_CONCURRENCY:
#         try:
#             result, token_usage = await generate_plot_analysis(
#                 movie,
#                 plot_synopsis=plot_synopsis,
#                 thematic_observations=thematic_observations,
#                 emotional_observations=None,
#                 provider=candidate.provider,
#                 model=candidate.model,
#                 system_prompt=candidate.system_prompt,
#                 response_format=candidate.response_format,
#                 **candidate.kwargs,
#             )
#             cost = _compute_cost(
#                 token_usage.model, token_usage.input_tokens, token_usage.output_tokens,
#             )
#             return {
#                 "candidate": candidate.label,
#                 "label": candidate.label,
#                 "model": candidate.model,
#                 "result": result.model_dump(),
#                 "input_tokens": token_usage.input_tokens,
#                 "output_tokens": token_usage.output_tokens,
#                 "cost_usd": cost,
#             }
#         except Exception as e:
#             print(f"  ✗ {candidate.label} for {movie.title_with_year()}: {type(e).__name__}: {e}")
#             return {
#                 "candidate": candidate.label,
#                 "label": candidate.label,
#                 "model": candidate.model,
#                 "result": None,
#                 "error": f"{type(e).__name__}: {e}",
#                 "input_tokens": None,
#                 "output_tokens": None,
#                 "cost_usd": None,
#             }


# def _pa_merge_results(
#     existing_results: list[dict],
#     new_results: list[dict],
# ) -> list[dict]:
#     """Merge new results into existing, keyed by label."""
#     merged = {r["label"]: r for r in existing_results}
#     for r in new_results:
#         merged[r["label"]] = r
#     seen = set()
#     ordered = []
#     for r in existing_results:
#         ordered.append(merged[r["label"]])
#         seen.add(r["label"])
#     for r in new_results:
#         if r["label"] not in seen:
#             ordered.append(merged[r["label"]])
#             seen.add(r["label"])
#     return ordered


# # ── Run all buckets ──────────────────────────────────────────────────────────

# _pa_eval_dir = project_root / "movie_ingestion" / "metadata_generation" / "evaluation_data"
# _pa_eval_dir.mkdir(parents=True, exist_ok=True)

# _pa_total = sum(len(b["tmdb_ids"]) for b in _pa_config["buckets"].values())
# _pa_idx = 0
# _pa_errors = 0

# for bucket_name, bucket in _pa_config["buckets"].items():
#     print(f"\n{'═' * 60}")
#     print(f"Bucket: {bucket_name} ({len(bucket['tmdb_ids'])} movies)")
#     print(f"{'═' * 60}")

#     _bucket_tasks = []
#     _bucket_movie_info = []

#     for tmdb_id in bucket["tmdb_ids"]:
#         if tmdb_id not in _pa_movies:
#             print(f"  SKIP: tmdb_id {tmdb_id} not loaded")
#             continue
#         m = _pa_movies[tmdb_id]
#         movie = m["movie"]
#         title = movie.title_with_year()

#         # 3 candidates per movie, no emotional_observations for any
#         movie_tasks = []
#         for candidate in PLOT_ANALYSIS_CANDIDATES:
#             movie_tasks.append(_pa_generate_one(
#                 movie, m["plot_synopsis"], m["thematic_observations"],
#                 candidate,
#             ))

#         _bucket_tasks.append(movie_tasks)
#         _bucket_movie_info.append((tmdb_id, title))

#     # Fire all tasks for the bucket concurrently (semaphore limits concurrency)
#     all_coros = [coro for movie_tasks in _bucket_tasks for coro in movie_tasks]
#     all_results = await asyncio.gather(*all_coros)

#     # Partition results back to per-movie groups (3 results per movie)
#     results_per_movie = []
#     offset = 0
#     for movie_tasks in _bucket_tasks:
#         n = len(movie_tasks)
#         results_per_movie.append(all_results[offset:offset + n])
#         offset += n

#     # Save per-movie results
#     for (tmdb_id, title), movie_results in zip(_bucket_movie_info, results_per_movie):
#         _pa_idx += 1

#         m = _pa_movies[tmdb_id]
#         user_prompt = build_plot_analysis_user_prompt(
#             m["movie"], m["plot_synopsis"], m["thematic_observations"],
#             None,
#         )

#         successes = sum(1 for r in movie_results if r["result"] is not None)
#         failures = len(movie_results) - successes
#         _pa_errors += failures

#         total_cost = sum(r["cost_usd"] or 0 for r in movie_results)

#         status = "✓" if failures == 0 else f"⚠ {failures} failed"
#         print(f"  [{_pa_idx}/{_pa_total}] {title}: {status}, ${total_cost:.4f}")

#         # Load existing file and merge
#         out_path = _pa_eval_dir / f"plot_analysis_{tmdb_id}.json"
#         if out_path.exists():
#             existing = _json.loads(out_path.read_text())
#             merged = _pa_merge_results(
#                 existing.get("candidate_results", []), list(movie_results),
#             )
#         else:
#             merged = list(movie_results)

#         movie_output = {
#             "tmdb_id": tmdb_id,
#             "title": title,
#             "bucket": bucket_name,
#             "user_prompt": user_prompt,
#             "candidate_results": merged,
#         }
#         out_path.write_text(
#             _json.dumps(movie_output, indent=2, ensure_ascii=False),
#         )

# # ── Summary ──────────────────────────────────────────────────────────────────

# _pa_total_cost = 0
# _pa_total_runs = 0
# for bucket_name, bucket in _pa_config["buckets"].items():
#     for tmdb_id in bucket["tmdb_ids"]:
#         out_path = _pa_eval_dir / f"plot_analysis_{tmdb_id}.json"
#         if out_path.exists():
#             data = _json.loads(out_path.read_text())
#             for r in data["candidate_results"]:
#                 _pa_total_runs += 1
#                 _pa_total_cost += r.get("cost_usd") or 0

# print(f"\n{'═' * 60}")
# print(f"COMPLETE: {_pa_total} movies × 3 runs = {_pa_total_runs} generations")
# print(f"Total cost: ${_pa_total_cost:.4f}")
# print(f"Errors: {_pa_errors}")
# print(f"Results saved to: {_pa_eval_dir}/plot_analysis_*.json")

In [19]:
# # ── Viewer Experience (Wave 2) — Round 4: GPO-Only Narrative ─────────────────
# # Tests whether generalized_plot_overview alone is a sufficient narrative
# # source — strips plot_summary and all raw plot fallbacks so the only
# # narrative the model sees is GPO (or nothing if GPO doesn't exist).
# #
# # Candidates:
# #   gpo-only: gpt-5-mini, minimal reasoning, low verbosity,
# #             justifications schema, narrative restricted to GPO
# #
# # Parallelism: all movies launched concurrently (semaphore caps in-flight)

# import json as _json
# import asyncio
# import time
# from dataclasses import replace as dc_replace

# from movie_ingestion.metadata_generation.generators.viewer_experience import (
#     build_viewer_experience_user_prompt,
#     generate_viewer_experience,
# )
# from movie_ingestion.metadata_generation.schemas import (
#     ViewerExperienceWithJustificationsOutput,
# )
# from movie_ingestion.metadata_generation.prompts.viewer_experience import (
#     SYSTEM_PROMPT_WITH_JUSTIFICATIONS as VE_SYSTEM_PROMPT_JUSTIFICATIONS,
# )

# # ── Candidate ────────────────────────────────────────────────────────────────
# # Single candidate: same model/params as production winner, but narrative
# # input restricted to generalized_plot_overview only.

# ve_candidates = [
#     PlaygroundCandidate(
#         label="gpo-only",
#         provider=LLMProvider.OPENAI,
#         model="gpt-5-mini",
#         system_prompt=VE_SYSTEM_PROMPT_JUSTIFICATIONS,
#         response_format=ViewerExperienceWithJustificationsOutput,
#         kwargs={"reasoning_effort": "minimal", "verbosity": "low"},
#     ),
# ]

# # ── Load bucket definitions ──────────────────────────────────────────────────

# _ve_buckets_path = project_root / "ingestion_data" / "viewer_experience_eval_buckets.json"
# with open(_ve_buckets_path) as _f:
#     _ve_buckets = _json.load(_f)

# # Collect all tmdb_ids across all buckets, preserving bucket membership
# ve_eval_groups: dict[str, list[int]] = {}
# ve_all_ids: list[int] = []
# for bucket_name, bucket_data in _ve_buckets["buckets"].items():
#     ids = [m["tmdb_id"] for m in bucket_data["movies"]]
#     ve_eval_groups[bucket_name] = ids
#     ve_all_ids.extend(ids)

# # Deduplicate while preserving order (a movie shouldn't appear in multiple buckets,
# # but guard against it)
# ve_all_ids = list(dict.fromkeys(ve_all_ids))
# print(f"Loaded {len(ve_eval_groups)} buckets, {len(ve_all_ids)} unique movies total")
# for bname, bids in ve_eval_groups.items():
#     print(f"  {bname}: {len(bids)} movies")

# # ── Build MovieInputData + load Wave 1 metadata for each movie ──────────────

# _ve_db = sqlite3.connect(str(project_root / "ingestion_data" / "tracker.db"))
# _ve_db.row_factory = sqlite3.Row

# _ve_placeholders = ",".join("?" * len(ve_all_ids))

# # TMDB base data
# _ve_tmdb_rows = {
#     row["tmdb_id"]: row
#     for row in _ve_db.execute(
#         f"SELECT tmdb_id, title, release_date, maturity_rating "
#         f"FROM tmdb_data WHERE tmdb_id IN ({_ve_placeholders})",
#         ve_all_ids,
#     ).fetchall()
# }

# # IMDB scraped data
# _ve_imdb_rows = {
#     row["tmdb_id"]: deserialize_imdb_row(row)
#     for row in _ve_db.execute(
#         f"SELECT * FROM imdb_data WHERE tmdb_id IN ({_ve_placeholders})",
#         ve_all_ids,
#     ).fetchall()
# }

# # Wave 1 generated metadata (plot_events, reception, plot_analysis)
# _ve_gen_rows = {
#     row["tmdb_id"]: row
#     for row in _ve_db.execute(
#         f"SELECT tmdb_id, plot_events, reception, plot_analysis "
#         f"FROM generated_metadata WHERE tmdb_id IN ({_ve_placeholders})",
#         ve_all_ids,
#     ).fetchall()
# }
# _ve_db.close()


# def _build_ve_movie(tmdb_id: int) -> MovieInputData | None:
#     """Build MovieInputData from tmdb + imdb rows."""
#     tmdb = _ve_tmdb_rows.get(tmdb_id)
#     if tmdb is None:
#         print(f"WARNING: tmdb_id {tmdb_id} not found in tmdb_data, skipping")
#         return None
#     imdb = _ve_imdb_rows.get(tmdb_id, {})
#     release_date = tmdb["release_date"] or ""
#     release_year = int(release_date[:4]) if release_date else None
#     maturity_rating = imdb.get("maturity_rating") or tmdb["maturity_rating"] or ""
#     return MovieInputData(
#         tmdb_id=tmdb_id,
#         title=tmdb["title"],
#         release_year=release_year,
#         overview=imdb.get("overview") or "",
#         genres=imdb.get("genres") or [],
#         plot_synopses=imdb.get("synopses") or [],
#         plot_summaries=imdb.get("plot_summaries") or [],
#         plot_keywords=imdb.get("plot_keywords") or [],
#         overall_keywords=imdb.get("overall_keywords") or [],
#         featured_reviews=imdb.get("featured_reviews") or [],
#         reception_summary=imdb.get("reception_summary"),
#         audience_reception_attributes=imdb.get("review_themes") or [],
#         maturity_rating=maturity_rating,
#         maturity_reasoning=imdb.get("maturity_reasoning") or [],
#         parental_guide_items=imdb.get("parental_guide_items") or [],
#     )


# def _extract_wave1_inputs(tmdb_id: int) -> dict:
#     """Extract Wave 1 metadata fields needed by generate_viewer_experience.

#     Returns a dict of kwargs ready to spread into the generator call.
#     """
#     gen_row = _ve_gen_rows.get(tmdb_id)
#     if gen_row is None:
#         return {}

#     inputs = {}

#     # plot_events → plot_summary
#     pe_raw = gen_row["plot_events"]
#     if pe_raw:
#         pe = _json.loads(pe_raw)
#         inputs["plot_summary"] = pe.get("plot_summary")

#     # reception → emotional/craft/thematic observations
#     rec_raw = gen_row["reception"]
#     if rec_raw:
#         rec = _json.loads(rec_raw)
#         inputs["emotional_observations"] = rec.get("emotional_observations")
#         inputs["craft_observations"] = rec.get("craft_observations")
#         inputs["thematic_observations"] = rec.get("thematic_observations")

#     # plot_analysis → generalized_plot_overview, genre_signatures, character_arcs
#     pa_raw = gen_row["plot_analysis"]
#     if pa_raw:
#         pa = _json.loads(pa_raw)
#         inputs["generalized_plot_overview"] = pa.get("generalized_plot_overview")
#         inputs["genre_signatures"] = pa.get("genre_signatures")
#         # character_arcs are stored as list of dicts with arc_transformation_label;
#         # the generator expects a list of label strings
#         raw_arcs = pa.get("character_arcs")
#         if raw_arcs and isinstance(raw_arcs, list):
#             inputs["character_arcs"] = [
#                 arc["arc_transformation_label"]
#                 for arc in raw_arcs
#                 if isinstance(arc, dict) and "arc_transformation_label" in arc
#             ]

#     return inputs


# # Build MovieInputData and Wave 1 inputs for all evaluation movies
# ve_movies: dict[int, MovieInputData] = {}
# ve_wave1_inputs: dict[int, dict] = {}
# for tid in ve_all_ids:
#     m = _build_ve_movie(tid)
#     if m is not None:
#         ve_movies[tid] = m
#         ve_wave1_inputs[tid] = _extract_wave1_inputs(tid)

# print(f"\nBuilt {len(ve_movies)} MovieInputData objects with Wave 1 inputs")
# print(f"Using {len(ve_candidates)} candidate(s):")
# for c in ve_candidates:
#     print(f"  - {c.label}")

# # ── GPO-only narrative restriction ───────────────────────────────────────────
# # Strip plot_summary from wave1 and zero out raw plot fields on the movie
# # object so best_plot_fallback() returns None. This forces the narrative
# # fallback logic to only see generalized_plot_overview (or nothing).


# def _restrict_to_gpo(
#     movie: MovieInputData, wave1: dict,
# ) -> tuple[MovieInputData, dict]:
#     """Return modified movie + wave1 with narrative restricted to GPO only."""
#     # Remove plot_summary so fallback step 1 sees nothing
#     restricted_wave1 = {k: v for k, v in wave1.items() if k != "plot_summary"}

#     # Zero out raw plot sources so best_plot_fallback() returns None
#     restricted_movie = dc_replace(
#         movie,
#         plot_synopses=[],
#         plot_summaries=[],
#         overview="",
#     )
#     return restricted_movie, restricted_wave1


# # ── Generation helper ────────────────────────────────────────────────────────

# _VE_CONCURRENCY = asyncio.Semaphore(50)


# async def _ve_generate_for_candidate(
#     movie: MovieInputData,
#     wave1: dict,
#     candidate: PlaygroundCandidate,
# ) -> dict:
#     """Generate viewer experience for one (movie, candidate) pair.

#     Restricts narrative input to generalized_plot_overview only.
#     """
#     effective_movie, effective_wave1 = _restrict_to_gpo(movie, wave1)

#     async with _VE_CONCURRENCY:
#         try:
#             result, token_usage = await generate_viewer_experience(
#                 effective_movie,
#                 provider=candidate.provider,
#                 model=candidate.model,
#                 system_prompt=candidate.system_prompt,
#                 response_format=candidate.response_format,
#                 **effective_wave1,
#                 **candidate.kwargs,
#             )
#             cost = _compute_cost(
#                 token_usage.model, token_usage.input_tokens, token_usage.output_tokens,
#             )
#             cost_str = f"${cost:.6f}" if cost is not None else "n/a"
#             print(
#                 f"  ✓ {candidate.label}: "
#                 f"{token_usage.input_tokens} in / {token_usage.output_tokens} out, "
#                 f"cost={cost_str}"
#             )
#             return {
#                 "candidate": candidate.label,
#                 "model": candidate.model,
#                 "result": result.model_dump(),
#                 "input_tokens": token_usage.input_tokens,
#                 "output_tokens": token_usage.output_tokens,
#                 "cost_usd": cost,
#             }
#         except Exception as e:
#             print(f"  ✗ {candidate.label}: {type(e).__name__}: {e}")
#             return {
#                 "candidate": candidate.label,
#                 "model": candidate.model,
#                 "result": None,
#                 "error": f"{type(e).__name__}: {e}",
#                 "input_tokens": None,
#                 "output_tokens": None,
#                 "cost_usd": None,
#             }


# def _ve_merge_results(
#     existing_results: dict[str, dict],
#     new_results: dict[str, dict],
# ) -> dict[str, dict]:
#     """Merge new candidate results into existing ones, keyed by label."""
#     merged = dict(existing_results)
#     merged.update(new_results)
#     return merged


# # ── Launch all movies in parallel ────────────────────────────────────────────

# eval_dir = project_root / "movie_ingestion" / "metadata_generation" / "evaluation_data"
# eval_dir.mkdir(parents=True, exist_ok=True)

# # Build ordered list of (tmdb_id, movie, wave1) tuples
# _ve_movie_list = [
#     (tid, ve_movies[tid], ve_wave1_inputs.get(tid, {}))
#     for tid in ve_all_ids
#     if tid in ve_movies
# ]

# total_movies = len(_ve_movie_list)
# total_tasks = total_movies * len(ve_candidates)
# t_start = time.monotonic()

# print(f"Launching generation: {total_movies} movies × {len(ve_candidates)} candidate(s) = {total_tasks} tasks")
# print(f"Concurrency limit: {_VE_CONCURRENCY._value}\n")

# # Build ALL coroutines at once
# all_coros = []
# all_keys: list[tuple[int, str]] = []  # (tmdb_id, candidate_label)
# for tmdb_id, movie, wave1 in _ve_movie_list:
#     for candidate in ve_candidates:
#         all_keys.append((tmdb_id, candidate.label))
#         all_coros.append(
#             _ve_generate_for_candidate(movie, wave1, candidate)
#         )

# all_results = await asyncio.gather(*all_coros)

# # Group results by movie
# results_per_movie: dict[int, dict[str, dict]] = {}
# for (tmdb_id, cand_label), result in zip(all_keys, all_results):
#     results_per_movie.setdefault(tmdb_id, {})[cand_label] = result

# # Save each movie's results
# total_errors = 0
# total_cost = 0.0
# for tmdb_id, movie, wave1 in _ve_movie_list:
#     title = movie.title_with_year()
#     # Save the full-input user_prompt as the reference (unpruned)
#     user_prompt = build_viewer_experience_user_prompt(movie, **wave1)
#     candidate_dict = results_per_movie.get(tmdb_id, {})

#     movie_errors = sum(1 for r in candidate_dict.values() if r["result"] is None)
#     movie_cost = sum(r["cost_usd"] or 0 for r in candidate_dict.values())
#     total_errors += movie_errors
#     total_cost += movie_cost

#     status = "ok" if movie_errors == 0 else f"{movie_errors} failed"
#     print(f"  {title}: {status}, ${movie_cost:.4f}")

#     # Load existing file and merge (preserves results from prior rounds)
#     out_path = eval_dir / f"viewer_experience_{tmdb_id}.json"
#     if out_path.exists():
#         existing = _json.loads(out_path.read_text())
#         merged = _ve_merge_results(
#             existing.get("candidate_results", {}), candidate_dict,
#         )
#     else:
#         merged = candidate_dict

#     movie_output = {
#         "tmdb_id": tmdb_id,
#         "title": title,
#         "user_prompt": user_prompt,
#         "candidate_results": merged,
#     }
#     out_path.write_text(_json.dumps(movie_output, indent=2, ensure_ascii=False))

# elapsed = time.monotonic() - t_start
# print(f"\nDone: {total_movies} movies × {len(ve_candidates)} candidate(s)")
# print(f"Total cost: ${total_cost:.4f} | Errors: {total_errors}")
# print(f"Elapsed: {elapsed:.1f}s ({total_tasks / elapsed:.1f} gen/sec)")

In [20]:
# ── Watch Context (Wave 2) — Round 3: No Signal Sources ──────────────────────
# Single candidate: gpt-5-mini low reasoning with justifications, per-section
# "Signal sources" guidance removed from prompt. The model decides which inputs
# inform which sections, relying on the general cross-section guidance in the
# INPUTS block ("Each observation field is strongest for its namesake section
# but may inform any section. Use your judgment").
# Tests whether open routing produces equivalent or better section assignment
# vs. prescriptive per-section routing
# (see r3-no-signal-sources in eval guide — H3 hypothesis test).
#
# Candidate:
#   r3-no-signal-sources — low reasoning, justifications, no signal sources
#
# Rate target: 120 gen/sec
# Parallelism: all movies launched concurrently (semaphore caps in-flight)

import json as _wc_json
import asyncio
import time

from movie_ingestion.metadata_generation.generators.watch_context import (
    build_watch_context_user_prompt,
    generate_watch_context,
)
from movie_ingestion.metadata_generation.schemas import (
    WatchContextWithIdentityNoteOutput,
)
from movie_ingestion.metadata_generation.prompts.watch_context import (
    SYSTEM_PROMPT_WITH_JUSTIFICATIONS as WC_SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
)

# ── Candidates ───────────────────────────────────────────────────────────────

wc_candidates = [
    PlaygroundCandidate(
        "r4-identity-note-low",
        LLMProvider.OPENAI,
        "gpt-5-mini",
        {"reasoning_effort": "low", "verbosity": "low"},
        WC_SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
        WatchContextWithIdentityNoteOutput,
    ),
    PlaygroundCandidate(
        "r4-identity-note-minimal",
        LLMProvider.OPENAI,
        "gpt-5-mini",
        {"reasoning_effort": "minimal", "verbosity": "low"},
        WC_SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
        WatchContextWithIdentityNoteOutput,
    ),
]

# ── Load bucket definitions ──────────────────────────────────────────────────
# watch_context_eval_buckets.json is a flat dict: bucket_name → list[tmdb_id]

_wc_buckets_path = project_root / "ingestion_data" / "watch_context_eval_buckets.json"
with open(_wc_buckets_path) as _f:
    _wc_buckets = _wc_json.load(_f)

wc_eval_groups: dict[str, list[int]] = {}
wc_all_ids: list[int] = []
for bucket_name, id_list in _wc_buckets.items():
    wc_eval_groups[bucket_name] = id_list
    wc_all_ids.extend(id_list)

# Deduplicate while preserving order
wc_all_ids = list(dict.fromkeys(wc_all_ids))
print(f"Loaded {len(wc_eval_groups)} buckets, {len(wc_all_ids)} unique movies total")
for bname, bids in wc_eval_groups.items():
    print(f"  {bname}: {len(bids)} movies")

# ── Build MovieInputData + load Wave 1 metadata for each movie ──────────────

_wc_db = sqlite3.connect(str(project_root / "ingestion_data" / "tracker.db"))
_wc_db.row_factory = sqlite3.Row

_wc_placeholders = ",".join("?" * len(wc_all_ids))

# TMDB base data
_wc_tmdb_rows = {
    row["tmdb_id"]: row
    for row in _wc_db.execute(
        f"SELECT tmdb_id, title, release_date, maturity_rating "
        f"FROM tmdb_data WHERE tmdb_id IN ({_wc_placeholders})",
        wc_all_ids,
    ).fetchall()
}

# IMDB scraped data
_wc_imdb_rows = {
    row["tmdb_id"]: deserialize_imdb_row(row)
    for row in _wc_db.execute(
        f"SELECT * FROM imdb_data WHERE tmdb_id IN ({_wc_placeholders})",
        wc_all_ids,
    ).fetchall()
}

# Wave 1 generated metadata (reception for observations, plot_analysis for genre_signatures)
_wc_gen_rows = {
    row["tmdb_id"]: row
    for row in _wc_db.execute(
        f"SELECT tmdb_id, reception, plot_analysis "
        f"FROM generated_metadata WHERE tmdb_id IN ({_wc_placeholders})",
        wc_all_ids,
    ).fetchall()
}
_wc_db.close()


def _build_wc_movie(tmdb_id: int) -> MovieInputData | None:
    """Build MovieInputData from tmdb + imdb rows."""
    tmdb = _wc_tmdb_rows.get(tmdb_id)
    if tmdb is None:
        print(f"WARNING: tmdb_id {tmdb_id} not found in tmdb_data, skipping")
        return None
    imdb = _wc_imdb_rows.get(tmdb_id, {})
    release_date = tmdb["release_date"] or ""
    release_year = int(release_date[:4]) if release_date else None
    maturity_rating = imdb.get("maturity_rating") or tmdb["maturity_rating"] or ""
    return MovieInputData(
        tmdb_id=tmdb_id,
        title=tmdb["title"],
        release_year=release_year,
        overview=imdb.get("overview") or "",
        genres=imdb.get("genres") or [],
        plot_synopses=imdb.get("synopses") or [],
        plot_summaries=imdb.get("plot_summaries") or [],
        plot_keywords=imdb.get("plot_keywords") or [],
        overall_keywords=imdb.get("overall_keywords") or [],
        featured_reviews=imdb.get("featured_reviews") or [],
        reception_summary=imdb.get("reception_summary"),
        audience_reception_attributes=imdb.get("review_themes") or [],
        maturity_rating=maturity_rating,
        maturity_reasoning=imdb.get("maturity_reasoning") or [],
        parental_guide_items=imdb.get("parental_guide_items") or [],
    )


def _extract_wc_wave1_inputs(tmdb_id: int) -> dict:
    """Extract Wave 1 metadata fields needed by generate_watch_context.

    Returns a kwargs dict with: genre_signatures, emotional_observations,
    craft_observations, thematic_observations.
    """
    gen_row = _wc_gen_rows.get(tmdb_id)
    if gen_row is None:
        return {}

    inputs = {}

    # reception → emotional/craft/thematic observations
    rec_raw = gen_row["reception"]
    if rec_raw:
        rec = _wc_json.loads(rec_raw)
        inputs["emotional_observations"] = rec.get("emotional_observations")
        inputs["craft_observations"] = rec.get("craft_observations")
        inputs["thematic_observations"] = rec.get("thematic_observations")

    # plot_analysis → genre_signatures
    pa_raw = gen_row["plot_analysis"]
    if pa_raw:
        pa = _wc_json.loads(pa_raw)
        inputs["genre_signatures"] = pa.get("genre_signatures")

    return inputs


# Build MovieInputData and Wave 1 inputs for all evaluation movies
wc_movies: dict[int, MovieInputData] = {}
wc_wave1_inputs: dict[int, dict] = {}
for tid in wc_all_ids:
    m = _build_wc_movie(tid)
    if m is not None:
        wc_movies[tid] = m
        wc_wave1_inputs[tid] = _extract_wc_wave1_inputs(tid)

print(f"\nBuilt {len(wc_movies)} MovieInputData objects with Wave 1 inputs")
print(f"Using {len(wc_candidates)} candidate(s):")
for c in wc_candidates:
    print(f"  - {c.label}")

# ── Rate limiter ─────────────────────────────────────────────────────────────
# 120 gen/sec total across candidates → semaphore per candidate
# Token-bucket interval: 1/120 sec between launches

_WC_RATE = 120
_WC_SEMAPHORE = asyncio.Semaphore(_WC_RATE // len(wc_candidates))
_WC_INTERVAL = 1.0 / _WC_RATE
_wc_last_launch = 0.0


async def _wc_generate_for_candidate(
    movie: MovieInputData,
    wave1: dict,
    candidate: PlaygroundCandidate,
) -> dict:
    """Generate watch context for one (movie, candidate) pair with rate limiting."""
    global _wc_last_launch

    # Token-bucket: wait until enough time has passed since last launch
    now = time.monotonic()
    wait = _WC_INTERVAL - (now - _wc_last_launch)
    if wait > 0:
        await asyncio.sleep(wait)
    _wc_last_launch = time.monotonic()

    async with _WC_SEMAPHORE:
        try:
            # Build kwargs for the generator: wave1 inputs + candidate overrides
            gen_kwargs = dict(wave1)
            gen_kwargs["provider"] = candidate.provider
            gen_kwargs["model"] = candidate.model
            if candidate.system_prompt is not None:
                gen_kwargs["system_prompt"] = candidate.system_prompt
            if candidate.response_format is not None:
                gen_kwargs["response_format"] = candidate.response_format
            gen_kwargs.update(candidate.kwargs)

            result, token_usage = await generate_watch_context(movie, **gen_kwargs)
            cost = _compute_cost(
                token_usage.model, token_usage.input_tokens, token_usage.output_tokens,
            )
            cost_str = f"${cost:.6f}" if cost is not None else "n/a"
            print(
                f"  ✓ {candidate.label}: "
                f"{token_usage.input_tokens} in / {token_usage.output_tokens} out, "
                f"cost={cost_str}"
            )
            return {
                "candidate": candidate.label,
                "model": candidate.model,
                "result": result.model_dump(),
                "input_tokens": token_usage.input_tokens,
                "output_tokens": token_usage.output_tokens,
                "cost_usd": cost,
            }
        except Exception as e:
            print(f"  ✗ {candidate.label}: {type(e).__name__}: {e}")
            return {
                "candidate": candidate.label,
                "model": candidate.model,
                "result": None,
                "error": f"{type(e).__name__}: {e}",
                "input_tokens": None,
                "output_tokens": None,
                "cost_usd": None,
            }


def _wc_merge_results(
    existing_results: dict[str, dict],
    new_results: dict[str, dict],
) -> dict[str, dict]:
    """Merge new candidate results into existing ones, keyed by label."""
    merged = dict(existing_results)
    merged.update(new_results)
    return merged


# ── Launch all movies in parallel ────────────────────────────────────────────

wc_eval_dir = project_root / "movie_ingestion" / "metadata_generation" / "evaluation_data"
wc_eval_dir.mkdir(parents=True, exist_ok=True)

# Build ordered list of (tmdb_id, movie, wave1) tuples
_wc_movie_list = [
    (tid, wc_movies[tid], wc_wave1_inputs.get(tid, {}))
    for tid in wc_all_ids
    if tid in wc_movies
]

wc_total_movies = len(_wc_movie_list)
wc_total_tasks = wc_total_movies * len(wc_candidates)
wc_t_start = time.monotonic()

print(f"Launching generation: {wc_total_movies} movies × {len(wc_candidates)} candidate(s) = {wc_total_tasks} tasks")
print(f"Concurrency limit: {_WC_SEMAPHORE._value} | Rate: {_WC_RATE} gen/sec\n")

# Build ALL coroutines at once
wc_all_coros = []
wc_all_keys: list[tuple[int, str]] = []  # (tmdb_id, candidate_label)
for tmdb_id, movie, wave1 in _wc_movie_list:
    for candidate in wc_candidates:
        wc_all_keys.append((tmdb_id, candidate.label))
        wc_all_coros.append(
            _wc_generate_for_candidate(movie, wave1, candidate)
        )

wc_all_results = await asyncio.gather(*wc_all_coros)

# Group results by movie
wc_results_per_movie: dict[int, dict[str, dict]] = {}
for (tmdb_id, cand_label), result in zip(wc_all_keys, wc_all_results):
    wc_results_per_movie.setdefault(tmdb_id, {})[cand_label] = result

# Save each movie's results
wc_total_errors = 0
wc_total_cost = 0.0
for tmdb_id, movie, wave1 in _wc_movie_list:
    title = movie.title_with_year()
    # Save the full-input user_prompt as the reference (using default schema prompt)
    user_prompt = build_watch_context_user_prompt(movie, **wave1)
    candidate_dict = wc_results_per_movie.get(tmdb_id, {})

    movie_errors = sum(1 for r in candidate_dict.values() if r["result"] is None)
    movie_cost = sum(r["cost_usd"] or 0 for r in candidate_dict.values())
    wc_total_errors += movie_errors
    wc_total_cost += movie_cost

    status = "ok" if movie_errors == 0 else f"{movie_errors} failed"
    print(f"  {title}: {status}, ${movie_cost:.4f}")

    # Load existing file and merge (preserves results from prior rounds)
    wc_out_path = wc_eval_dir / f"watch_context_{tmdb_id}.json"
    if wc_out_path.exists():
        existing = _wc_json.loads(wc_out_path.read_text())
        merged = _wc_merge_results(
            existing.get("candidate_results", {}), candidate_dict,
        )
    else:
        merged = candidate_dict

    movie_output = {
        "tmdb_id": tmdb_id,
        "title": title,
        "user_prompt": user_prompt,
        "candidate_results": merged,
    }
    wc_out_path.write_text(_wc_json.dumps(movie_output, indent=2, ensure_ascii=False))

wc_elapsed = time.monotonic() - wc_t_start
print(f"\nDone: {wc_total_movies} movies × {len(wc_candidates)} candidate(s)")
print(f"Total cost: ${wc_total_cost:.4f} | Errors: {wc_total_errors}")
print(f"Elapsed: {wc_elapsed:.1f}s ({wc_total_tasks / wc_elapsed:.1f} gen/sec)")

Loaded 6 buckets, 50 unique movies total
  gold_standard: 8 movies
  mid_observations: 8 movies
  sparse_full_coverage: 8 movies
  documentary_nonfiction: 8 movies
  maturity_edge: 8 movies
  challenging_identity: 10 movies

Built 50 MovieInputData objects with Wave 1 inputs
Using 2 candidate(s):
  - r4-identity-note-low
  - r4-identity-note-minimal
Launching generation: 50 movies × 2 candidate(s) = 100 tasks
Concurrency limit: 60 | Rate: 120 gen/sec

  ✓ r4-identity-note-minimal: 2651 in / 359 out, cost=$0.001381
  ✓ r4-identity-note-minimal: 2663 in / 379 out, cost=$0.001424
  ✓ r4-identity-note-minimal: 2651 in / 357 out, cost=$0.001377
  ✓ r4-identity-note-minimal: 2741 in / 387 out, cost=$0.001459
  ✓ r4-identity-note-minimal: 2676 in / 376 out, cost=$0.001421
  ✓ r4-identity-note-minimal: 2755 in / 376 out, cost=$0.001441
  ✓ r4-identity-note-minimal: 2750 in / 397 out, cost=$0.001481
  ✓ r4-identity-note-minimal: 2667 in / 402 out, cost=$0.001471
  ✓ r4-identity-note-minimal: 26

In [ ]:
# # ── Narrative Techniques (Wave 2) — Round 3 ──────────────────────────────────
# # Broad model sweep: one candidate per MODEL_PRICING model (minus gpt-oss,
# # llama-4-scout, gpt-5-nano). Two gpt-5-mini variants (low + minimal).
# # All candidates use the justifications prompt/schema.
# #
# # Processes one movie at a time (all candidates concurrently per movie)
# # to stay within rate limits on non-OpenAI providers.
# #
# # Results are MERGED into existing per-movie JSON files so multiple
# # runs (baseline, ablations) accumulate in the same output files.

# import json as _json
# import asyncio
# import copy
# import time
# from movie_ingestion.metadata_generation.generators.narrative_techniques import (
#     build_narrative_techniques_user_prompt,
#     generate_narrative_techniques,
# )
# from movie_ingestion.metadata_generation.prompts.narrative_techniques import (
#     SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
# )
# from movie_ingestion.metadata_generation.schemas import NarrativeTechniquesWithJustificationsOutput

# # ── Candidates ────────────────────────────────────────────────────────────────

# nt_candidates = [
#     PlaygroundCandidate(
#         "r3-gpt5mini-low",
#         LLMProvider.OPENAI,
#         "gpt-5-mini",
#         {"reasoning_effort": "low", "verbosity": "low"},
#         SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
#         NarrativeTechniquesWithJustificationsOutput,
#     ),
#     PlaygroundCandidate(
#         "r3-gpt5mini-minimal",
#         LLMProvider.OPENAI,
#         "gpt-5-mini",
#         {"reasoning_effort": "minimal", "verbosity": "low"},
#         SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
#         NarrativeTechniquesWithJustificationsOutput,
#     ),
#     PlaygroundCandidate(
#         "r3-gpt54nano-med",
#         LLMProvider.OPENAI,
#         "gpt-5.4-nano",
#         {"reasoning_effort": "medium"},
#         SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
#         NarrativeTechniquesWithJustificationsOutput,
#     ),
#     # PlaygroundCandidate(
#     #     "r3-qwen35-flash",
#     #     LLMProvider.ALIBABA,
#     #     "qwen3.5-flash",
#     #     {"temperature": 0.2, "extra_body": {"enable_thinking": False}},
#     #     SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
#     #     NarrativeTechniquesWithJustificationsOutput,
#     # ),
#     PlaygroundCandidate(
#         "r3-kimi-k25",
#         LLMProvider.KIMI,
#         "kimi-k2.5",
#         {"enable_thinking": False},
#         SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
#         NarrativeTechniquesWithJustificationsOutput,
#     ),
#     PlaygroundCandidate(
#         "r3-gemini-25-flash",
#         LLMProvider.GEMINI,
#         "gemini-2.5-flash",
#         {"temperature": 0.2, "thinking_config": {"thinking_budget": 1024}},
#         SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
#         NarrativeTechniquesWithJustificationsOutput,
#     ),
#     PlaygroundCandidate(
#         "r3-gemini-25-flash-lite",
#         LLMProvider.GEMINI,
#         "gemini-2.5-flash-lite",
#         {"temperature": 0.2, "thinking_config": {"thinking_budget": 1024}},
#         SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
#         NarrativeTechniquesWithJustificationsOutput,
#     ),
#     PlaygroundCandidate(
#         "r3-gemini-31-flash-lite",
#         LLMProvider.GEMINI,
#         "gemini-3.1-flash-lite-preview",
#         {"temperature": 0.2, "thinking_config": {"thinking_budget": 1024}},
#         SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
#         NarrativeTechniquesWithJustificationsOutput,
#     ),
# ]

# # Labels that use overall_keywords only (plot_keywords cleared before generation)
# _NT_OVERALL_KW_LABELS: set[str] = set()

# # ── Load bucket definitions ──────────────────────────────────────────────────

# _nt_buckets_path = project_root / "ingestion_data" / "narrative_techniques_eval_buckets.json"
# with open(_nt_buckets_path) as _f:
#     _nt_buckets = _json.load(_f)

# # Collect all tmdb_ids across all buckets, preserving bucket membership
# nt_eval_groups: dict[str, list[int]] = {}
# nt_all_ids: list[int] = []
# for bucket_name, bucket_data in _nt_buckets["buckets"].items():
#     ids = [m["tmdb_id"] for m in bucket_data["movies"]]
#     nt_eval_groups[bucket_name] = ids
#     nt_all_ids.extend(ids)

# # Deduplicate while preserving order
# nt_all_ids = list(dict.fromkeys(nt_all_ids))
# print(f"Loaded {len(nt_eval_groups)} buckets, {len(nt_all_ids)} unique movies total")
# for bname, bids in nt_eval_groups.items():
#     print(f"  {bname}: {len(bids)} movies")

# # ── Build MovieInputData + load Wave 1 metadata for each movie ──────────────

# _nt_db = sqlite3.connect(str(project_root / "ingestion_data" / "tracker.db"))
# _nt_db.row_factory = sqlite3.Row

# _nt_placeholders = ",".join("?" * len(nt_all_ids))

# # TMDB base data
# _nt_tmdb_rows = {
#     row["tmdb_id"]: row
#     for row in _nt_db.execute(
#         f"SELECT tmdb_id, title, release_date, maturity_rating "
#         f"FROM tmdb_data WHERE tmdb_id IN ({_nt_placeholders})",
#         nt_all_ids,
#     ).fetchall()
# }

# # IMDB scraped data
# _nt_imdb_rows = {
#     row["tmdb_id"]: deserialize_imdb_row(row)
#     for row in _nt_db.execute(
#         f"SELECT * FROM imdb_data WHERE tmdb_id IN ({_nt_placeholders})",
#         nt_all_ids,
#     ).fetchall()
# }

# # Wave 1 generated metadata (plot_events, reception)
# _nt_gen_rows = {
#     row["tmdb_id"]: row
#     for row in _nt_db.execute(
#         f"SELECT tmdb_id, plot_events, reception "
#         f"FROM generated_metadata WHERE tmdb_id IN ({_nt_placeholders})",
#         nt_all_ids,
#     ).fetchall()
# }
# _nt_db.close()


# def _build_nt_movie(tmdb_id: int) -> MovieInputData | None:
#     """Build MovieInputData from tmdb + imdb rows."""
#     tmdb = _nt_tmdb_rows.get(tmdb_id)
#     if tmdb is None:
#         print(f"WARNING: tmdb_id {tmdb_id} not found in tmdb_data, skipping")
#         return None
#     imdb = _nt_imdb_rows.get(tmdb_id, {})
#     release_date = tmdb["release_date"] or ""
#     release_year = int(release_date[:4]) if release_date else None
#     maturity_rating = imdb.get("maturity_rating") or tmdb["maturity_rating"] or ""
#     return MovieInputData(
#         tmdb_id=tmdb_id,
#         title=tmdb["title"],
#         release_year=release_year,
#         overview=imdb.get("overview") or "",
#         genres=imdb.get("genres") or [],
#         plot_synopses=imdb.get("synopses") or [],
#         plot_summaries=imdb.get("plot_summaries") or [],
#         plot_keywords=imdb.get("plot_keywords") or [],
#         overall_keywords=imdb.get("overall_keywords") or [],
#         featured_reviews=imdb.get("featured_reviews") or [],
#         reception_summary=imdb.get("reception_summary"),
#         audience_reception_attributes=imdb.get("review_themes") or [],
#         maturity_rating=maturity_rating,
#         maturity_reasoning=imdb.get("maturity_reasoning") or [],
#         parental_guide_items=imdb.get("parental_guide_items") or [],
#     )


# def _extract_nt_wave1_inputs(tmdb_id: int) -> dict:
#     """Extract Wave 1 metadata fields needed by generate_narrative_techniques.

#     Returns a dict of kwargs: plot_summary and craft_observations.
#     """
#     gen_row = _nt_gen_rows.get(tmdb_id)
#     if gen_row is None:
#         return {}

#     inputs = {}

#     # plot_events → plot_summary
#     pe_raw = gen_row["plot_events"]
#     if pe_raw:
#         pe = _json.loads(pe_raw)
#         inputs["plot_summary"] = pe.get("plot_summary")

#     # reception → craft_observations
#     rec_raw = gen_row["reception"]
#     if rec_raw:
#         rec = _json.loads(rec_raw)
#         inputs["craft_observations"] = rec.get("craft_observations")

#     return inputs


# # Build MovieInputData and Wave 1 inputs for all evaluation movies
# nt_movies: dict[int, MovieInputData] = {}
# nt_wave1_inputs: dict[int, dict] = {}
# for tid in nt_all_ids:
#     m = _build_nt_movie(tid)
#     if m is not None:
#         nt_movies[tid] = m
#         nt_wave1_inputs[tid] = _extract_nt_wave1_inputs(tid)

# print(f"\nBuilt {len(nt_movies)} MovieInputData objects with Wave 1 inputs")
# print(f"Using {len(nt_candidates)} candidates:")
# for c in nt_candidates:
#     print(f"  - {c.label} ({c.model}, reasoning_effort={c.kwargs.get('reasoning_effort', 'default')})")

# # ── Generation helper ────────────────────────────────────────────────────────


# def _prepare_movie_for_candidate(
#     movie: MovieInputData, candidate: PlaygroundCandidate,
# ) -> MovieInputData:
#     """Return a (possibly modified) copy of the movie for this candidate."""
#     if candidate.label in _NT_OVERALL_KW_LABELS:
#         movie = copy.copy(movie)
#         movie.plot_keywords = []
#     return movie


# async def _nt_generate_for_candidate(
#     movie: MovieInputData,
#     wave1: dict,
#     candidate: PlaygroundCandidate,
# ) -> dict:
#     """Generate narrative techniques for one (movie, candidate) pair."""
#     movie = _prepare_movie_for_candidate(movie, candidate)

#     try:
#         result, token_usage = await generate_narrative_techniques(
#             movie,
#             provider=candidate.provider,
#             model=candidate.model,
#             system_prompt=candidate.system_prompt or SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
#             response_format=candidate.response_format or NarrativeTechniquesWithJustificationsOutput,
#             **wave1,
#             **candidate.kwargs,
#         )
#         cost = _compute_cost(
#             token_usage.model, token_usage.input_tokens, token_usage.output_tokens,
#         )
#         cost_str = f"${cost:.6f}" if cost is not None else "n/a"
#         print(
#             f"  ✓ {candidate.label}: "
#             f"{token_usage.input_tokens} in / {token_usage.output_tokens} out, "
#             f"cost={cost_str}"
#         )
#         return {
#             "candidate": candidate.label,
#             "model": candidate.model,
#             "result": result.model_dump(),
#             "input_tokens": token_usage.input_tokens,
#             "output_tokens": token_usage.output_tokens,
#             "cost_usd": cost,
#         }
#     except Exception as e:
#         print(f"  ✗ {candidate.label}: {type(e).__name__}: {e}")
#         return {
#             "candidate": candidate.label,
#             "model": candidate.model,
#             "result": None,
#             "error": f"{type(e).__name__}: {e}",
#             "input_tokens": None,
#             "output_tokens": None,
#             "cost_usd": None,
#         }


# # ── Generate one movie at a time, all candidates concurrently per movie ──────

# eval_dir = project_root / "movie_ingestion" / "metadata_generation" / "evaluation_data"
# eval_dir.mkdir(parents=True, exist_ok=True)

# # Build ordered list: flatten buckets so output groups by bucket
# _nt_movie_order: list[tuple[str, int]] = []  # (bucket_name, tmdb_id)
# _nt_seen: set[int] = set()
# for bucket_name, bucket_ids in nt_eval_groups.items():
#     for tid in bucket_ids:
#         if tid in nt_movies and tid not in _nt_seen:
#             _nt_movie_order.append((bucket_name, tid))
#             _nt_seen.add(tid)

# total_movies = len(_nt_movie_order)
# nt_total_errors = 0
# nt_total_cost = 0.0
# t_start = time.monotonic()

# print(f"Processing {total_movies} movies × {len(nt_candidates)} candidates "
#       f"(one movie at a time, candidates concurrent)\n")

# current_bucket = None
# for movie_idx, (bucket_name, tmdb_id) in enumerate(_nt_movie_order, 1):
#     if bucket_name != current_bucket:
#         current_bucket = bucket_name
#         print(f"\n{'='*60}")
#         print(f"Bucket: {bucket_name}")
#         print(f"{'='*60}")

#     movie = nt_movies[tmdb_id]
#     wave1 = nt_wave1_inputs.get(tmdb_id, {})
#     title = movie.title_with_year()
#     print(f"\n[{movie_idx}/{total_movies}] {title}")

#     # Run all candidates concurrently for this single movie
#     candidate_results = await asyncio.gather(*[
#         _nt_generate_for_candidate(movie, wave1, c) for c in nt_candidates
#     ])

#     # Index results by candidate label
#     new_candidates = {r["candidate"]: r for r in candidate_results}

#     # Tally errors and cost
#     movie_errors = sum(1 for r in new_candidates.values() if r["result"] is None)
#     movie_cost = sum(r["cost_usd"] or 0 for r in new_candidates.values())
#     nt_total_errors += movie_errors
#     nt_total_cost += movie_cost

#     status = "ok" if movie_errors == 0 else f"{movie_errors} failed"
#     print(f"  → {status}, ${movie_cost:.4f}")

#     # Load existing results and merge new candidates in
#     user_prompt = build_narrative_techniques_user_prompt(movie, **wave1)
#     out_path = eval_dir / f"narrative_techniques_{tmdb_id}.json"
#     if out_path.exists():
#         existing = _json.loads(out_path.read_text())
#         merged = existing.get("candidate_results", {})
#         merged.update(new_candidates)
#     else:
#         merged = new_candidates

#     movie_output = {
#         "tmdb_id": tmdb_id,
#         "title": title,
#         "user_prompt": user_prompt,
#         "candidate_results": merged,
#     }
#     out_path.write_text(_json.dumps(movie_output, indent=2, ensure_ascii=False))

# elapsed = time.monotonic() - t_start
# print(f"\nDone: {total_movies} movies × {len(nt_candidates)} candidates")
# print(f"Total cost: ${nt_total_cost:.4f} | Errors: {nt_total_errors}")
# print(f"Elapsed: {elapsed:.1f}s ({total_movies * len(nt_candidates) / elapsed:.1f} gen/sec)")

Loaded 7 buckets, 56 unique movies total
  gold_standard: 8 movies
  floor_tier1: 8 movies
  raw_fallback_standalone: 8 movies
  craft_standalone_rich: 8 movies
  craft_standalone_floor: 8 movies
  combined_path: 8 movies
  craft_threshold_edge: 8 movies

Built 56 MovieInputData objects with Wave 1 inputs
Using 7 candidates:
  - r3-gpt5mini-low (gpt-5-mini, reasoning_effort=low)
  - r3-gpt5mini-minimal (gpt-5-mini, reasoning_effort=minimal)
  - r3-gpt54nano-med (gpt-5.4-nano, reasoning_effort=medium)
  - r3-kimi-k25 (kimi-k2.5, reasoning_effort=default)
  - r3-gemini-25-flash (gemini-2.5-flash, reasoning_effort=default)
  - r3-gemini-25-flash-lite (gemini-2.5-flash-lite, reasoning_effort=default)
  - r3-gemini-31-flash-lite (gemini-3.1-flash-lite-preview, reasoning_effort=default)
Processing 56 movies × 7 candidates (one movie at a time, candidates concurrent)


Bucket: gold_standard

[1/56] The Shawshank Redemption (1994)
  ✓ r3-gemini-31-flash-lite: 3439 in / 556 out, cost=$0.001694


In [ ]:
# ── Source Of Inspiration (Wave 2) — model + justification comparison ──────
# Candidates:
#   gpt-5-mini minimal / low reasoning, low verbosity
#   gpt-5.4-nano medium reasoning
# Each model config runs with and without the justifications prompt/schema.
# Results are MERGED into existing per-movie JSON files when present.

import json as _json
import asyncio
import time
from movie_ingestion.metadata_generation.generators.source_of_inspiration import (
    build_source_of_inspiration_user_prompt,
    generate_source_of_inspiration,
)
from movie_ingestion.metadata_generation.prompts.source_of_inspiration import (
    SYSTEM_PROMPT as SOI_SYSTEM_PROMPT,
    SYSTEM_PROMPT_WITH_JUSTIFICATIONS as SOI_SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
)
from movie_ingestion.metadata_generation.schemas import (
    SourceOfInspirationOutput,
    SourceOfInspirationWithJustificationsOutput,
)

# ── Candidates ───────────────────────────────────────────────────────────────

soi_candidates = [
    PlaygroundCandidate(
        "gpt5mini-minimal",
        LLMProvider.OPENAI,
        "gpt-5-mini",
        {"reasoning_effort": "minimal", "verbosity": "low"},
    ),
    PlaygroundCandidate(
        "gpt5mini-minimal-just",
        LLMProvider.OPENAI,
        "gpt-5-mini",
        {"reasoning_effort": "minimal", "verbosity": "low"},
        SOI_SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
        SourceOfInspirationWithJustificationsOutput,
    ),
    PlaygroundCandidate(
        "gpt5mini-low",
        LLMProvider.OPENAI,
        "gpt-5-mini",
        {"reasoning_effort": "low", "verbosity": "low"},
    ),
    PlaygroundCandidate(
        "gpt5mini-low-just",
        LLMProvider.OPENAI,
        "gpt-5-mini",
        {"reasoning_effort": "low", "verbosity": "low"},
        SOI_SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
        SourceOfInspirationWithJustificationsOutput,
    ),
    PlaygroundCandidate(
        "gpt54nano-medium",
        LLMProvider.OPENAI,
        "gpt-5.4-nano",
        {"reasoning_effort": "medium", "verbosity": "low"},
    ),
    PlaygroundCandidate(
        "gpt54nano-medium-just",
        LLMProvider.OPENAI,
        "gpt-5.4-nano",
        {"reasoning_effort": "medium", "verbosity": "low"},
        SOI_SYSTEM_PROMPT_WITH_JUSTIFICATIONS,
        SourceOfInspirationWithJustificationsOutput,
    ),
]

# ── Load bucket definitions ─────────────────────────────────────────────────

_soi_buckets_path = project_root / "ingestion_data" / "source_of_inspiration_eval_buckets.json"
with open(_soi_buckets_path) as _f:
    _soi_buckets = _json.load(_f)

soi_eval_groups: dict[str, list[int]] = {}
soi_all_ids: list[int] = []
for bucket_name, bucket_data in _soi_buckets["buckets"].items():
    if "movies" in bucket_data:
        ids = [movie["tmdb_id"] for movie in bucket_data["movies"]]
    else:
        ids = bucket_data.get("tmdb_ids", [])
    soi_eval_groups[bucket_name] = ids
    soi_all_ids.extend(ids)

soi_all_ids = list(dict.fromkeys(soi_all_ids))
print(f"Loaded {len(soi_eval_groups)} buckets, {len(soi_all_ids)} unique movies total")
for bucket_name, bucket_ids in soi_eval_groups.items():
    print(f"  {bucket_name}: {len(bucket_ids)} movies")

# ── Build MovieInputData + load Wave 1 metadata for each movie ─────────────

_soi_db = sqlite3.connect(str(project_root / "ingestion_data" / "tracker.db"))
_soi_db.row_factory = sqlite3.Row

_soi_placeholders = ",".join("?" * len(soi_all_ids))

_soi_tmdb_rows = {
    row["tmdb_id"]: row
    for row in _soi_db.execute(
        f"SELECT tmdb_id, title, release_date, maturity_rating "
        f"FROM tmdb_data WHERE tmdb_id IN ({_soi_placeholders})",
        soi_all_ids,
    ).fetchall()
}

_soi_imdb_rows = {
    row["tmdb_id"]: deserialize_imdb_row(row)
    for row in _soi_db.execute(
        f"SELECT * FROM imdb_data WHERE tmdb_id IN ({_soi_placeholders})",
        soi_all_ids,
    ).fetchall()
}

_soi_gen_rows = {
    row["tmdb_id"]: row
    for row in _soi_db.execute(
        f"SELECT tmdb_id, reception "
        f"FROM generated_metadata WHERE tmdb_id IN ({_soi_placeholders})",
        soi_all_ids,
    ).fetchall()
}
_soi_db.close()


def _build_soi_movie(tmdb_id: int) -> MovieInputData | None:
    """Build MovieInputData from tmdb + imdb rows."""
    tmdb = _soi_tmdb_rows.get(tmdb_id)
    if tmdb is None:
        print(f"WARNING: tmdb_id {tmdb_id} not found in tmdb_data, skipping")
        return None
    imdb = _soi_imdb_rows.get(tmdb_id, {})
    release_date = tmdb["release_date"] or ""
    release_year = int(release_date[:4]) if release_date else None
    maturity_rating = imdb.get("maturity_rating") or tmdb["maturity_rating"] or ""
    return MovieInputData(
        tmdb_id=tmdb_id,
        title=tmdb["title"],
        release_year=release_year,
        overview=imdb.get("overview") or "",
        genres=imdb.get("genres") or [],
        plot_synopses=imdb.get("synopses") or [],
        plot_summaries=imdb.get("plot_summaries") or [],
        plot_keywords=imdb.get("plot_keywords") or [],
        overall_keywords=imdb.get("overall_keywords") or [],
        featured_reviews=imdb.get("featured_reviews") or [],
        reception_summary=imdb.get("reception_summary"),
        audience_reception_attributes=imdb.get("review_themes") or [],
        maturity_rating=maturity_rating,
        maturity_reasoning=imdb.get("maturity_reasoning") or [],
        parental_guide_items=imdb.get("parental_guide_items") or [],
    )


def _extract_soi_wave1_inputs(tmdb_id: int) -> dict:
    """Extract Wave 1 metadata fields needed by generate_source_of_inspiration."""
    source_material_hint = None
    gen_row = _soi_gen_rows.get(tmdb_id)
    if gen_row is not None and gen_row["reception"]:
        reception = _json.loads(gen_row["reception"])
        source_material_hint = reception.get("source_material_hint")
    return {"source_material_hint": source_material_hint}


soi_movies: dict[int, MovieInputData] = {}
soi_wave1_inputs: dict[int, dict] = {}
for tmdb_id in soi_all_ids:
    movie = _build_soi_movie(tmdb_id)
    if movie is not None:
        soi_movies[tmdb_id] = movie
        soi_wave1_inputs[tmdb_id] = _extract_soi_wave1_inputs(tmdb_id)

print(f"\nBuilt {len(soi_movies)} MovieInputData objects with Wave 1 inputs")
print(f"Using {len(soi_candidates)} candidates:")
for candidate in soi_candidates:
    print(
        f"  - {candidate.label} ({candidate.model}, "
        f"reasoning_effort={candidate.kwargs.get('reasoning_effort', 'default')})"
    )

# ── Generation helpers ──────────────────────────────────────────────────────

_SOI_CONCURRENCY = asyncio.Semaphore(60)


def _normalize_soi_candidate_results(candidate_results: object) -> dict[str, dict]:
    """Normalize old list-based or dict-based candidate results into a dict."""
    if isinstance(candidate_results, dict):
        return dict(candidate_results)
    if isinstance(candidate_results, list):
        normalized = {}
        for result in candidate_results:
            if isinstance(result, dict) and result.get("candidate"):
                normalized[result["candidate"]] = result
        return normalized
    return {}


async def _soi_generate_for_candidate(
    movie: MovieInputData,
    wave1: dict,
    candidate: PlaygroundCandidate,
) -> dict:
    """Generate source_of_inspiration for one (movie, candidate) pair."""
    async with _SOI_CONCURRENCY:
        try:
            result, token_usage = await generate_source_of_inspiration(
                movie,
                provider=candidate.provider,
                model=candidate.model,
                system_prompt=candidate.system_prompt or SOI_SYSTEM_PROMPT,
                response_format=candidate.response_format or SourceOfInspirationOutput,
                **wave1,
                **candidate.kwargs,
            )
            cost = _compute_cost(
                token_usage.model, token_usage.input_tokens, token_usage.output_tokens,
            )
            cost_str = f"${cost:.6f}" if cost is not None else "n/a"
            print(
                f"  ✓ {candidate.label}: "
                f"{token_usage.input_tokens} in / {token_usage.output_tokens} out, "
                f"cost={cost_str}"
            )
            return {
                "candidate": candidate.label,
                "model": candidate.model,
                "result": result.model_dump(),
                "input_tokens": token_usage.input_tokens,
                "output_tokens": token_usage.output_tokens,
                "cost_usd": cost,
            }
        except Exception as e:
            print(f"  ✗ {candidate.label}: {type(e).__name__}: {e}")
            return {
                "candidate": candidate.label,
                "model": candidate.model,
                "result": None,
                "error": f"{type(e).__name__}: {e}",
                "input_tokens": None,
                "output_tokens": None,
                "cost_usd": None,
            }


# ── Generate all movies × candidates in parallel ────────────────────────────

eval_dir = project_root / "movie_ingestion" / "metadata_generation" / "evaluation_data"
eval_dir.mkdir(parents=True, exist_ok=True)

_soi_coros = []
_soi_task_keys: list[tuple[int, str]] = []

for tmdb_id, movie in soi_movies.items():
    wave1 = soi_wave1_inputs.get(tmdb_id, {"source_material_hint": None})
    for candidate in soi_candidates:
        _soi_task_keys.append((tmdb_id, candidate.label))
        _soi_coros.append(_soi_generate_for_candidate(movie, wave1, candidate))

print(
    f"Launching {len(_soi_coros)} generation tasks "
    f"({len(soi_movies)} movies × {len(soi_candidates)} candidates)..."
)
t_start = time.monotonic()

_soi_all_results = await asyncio.gather(*_soi_coros)

elapsed = time.monotonic() - t_start
print(
    f"\nAll generations complete in {elapsed:.1f}s "
    f"({len(_soi_coros) / elapsed:.1f} generations/sec)"
)

# ── Group results by movie and merge into existing files ────────────────────

_soi_per_movie: dict[int, dict[str, dict]] = {}
for (tmdb_id, candidate_label), result in zip(_soi_task_keys, _soi_all_results):
    _soi_per_movie.setdefault(tmdb_id, {})[candidate_label] = result

soi_total_errors = 0
soi_total_cost = 0.0
for bucket_name, bucket_ids in soi_eval_groups.items():
    bucket_movies = [(tmdb_id, soi_movies[tmdb_id]) for tmdb_id in bucket_ids if tmdb_id in soi_movies]
    if not bucket_movies:
        continue
    print(f"\n{'=' * 60}")
    print(f"Bucket: {bucket_name} ({len(bucket_movies)} movies)")
    print(f"{'=' * 60}")

    for movie_idx, (tmdb_id, movie) in enumerate(bucket_movies, 1):
        title = movie.title_with_year()
        wave1 = soi_wave1_inputs.get(tmdb_id, {"source_material_hint": None})
        user_prompt = build_source_of_inspiration_user_prompt(movie, **wave1)
        new_candidates = _soi_per_movie.get(tmdb_id, {})

        movie_errors = sum(1 for result in new_candidates.values() if result["result"] is None)
        movie_cost = sum(result["cost_usd"] or 0 for result in new_candidates.values())
        soi_total_errors += movie_errors
        soi_total_cost += movie_cost

        status = "ok" if movie_errors == 0 else f"{movie_errors} failed"
        print(f"  [{movie_idx}/{len(bucket_movies)}] {title}: {status}, ${movie_cost:.4f}")

        out_path = eval_dir / f"source_of_inspiration_{tmdb_id}.json"
        if out_path.exists():
            existing_output = _json.loads(out_path.read_text())
            merged = _normalize_soi_candidate_results(existing_output.get("candidate_results"))
            merged.update(new_candidates)
        else:
            merged = new_candidates

        movie_output = {
            "tmdb_id": tmdb_id,
            "title": title,
            "user_prompt": user_prompt,
            "candidate_results": merged,
        }
        out_path.write_text(_json.dumps(movie_output, indent=2, ensure_ascii=False))

print(f"\nDone: {len(soi_movies)} movies × {len(soi_candidates)} candidates")
print(f"Total cost: ${soi_total_cost:.4f} | Errors: {soi_total_errors}")


In [ ]:
# Add code for generating plot keywords